In [ ]:
from functools import partial

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import optax
import torch
import torchvision.datasets as tvdatasets
import torchvision.transforms as tvtransforms
import xdg_base_dirs

import initialize
import mlp_wrapper
from jaxlnb import jaxlnb

In [ ]:
# Configuration

DATASETS_ROOT = xdg_base_dirs.xdg_data_home() / "lnb" / "mnist"

HPARAMS = {
    "seed": 0,
    "xinv": False,
    "batch_size": 1000,
    "mlp_kwargs": {
        "in_size": 28 * 28,
        "out_size": len(tvdatasets.MNIST.classes),
        "width_size": 800,
        "depth": 2 - 1,
    },
    "init": "glorot_normal",
    "lnb_kwargs": {
        "cg_ridge": 1.0,
    },
    "step_size": 0.5,
    "clip_norm": 1.0,
}

In [ ]:
# Load MNIST


def np_collate(batch):
    return jax.tree_util.tree_map(np.asarray, torch.utils.data.default_collate(batch))


def load_dataset(dataset):
    loader = torch.utils.data.DataLoader(dataset, batch_size=len(dataset), collate_fn=np_collate)
    images, labels = next(iter(loader))
    del loader
    return jnp.array(images), jnp.array(labels)


transforms = [tvtransforms.ToTensor(), tvtransforms.Lambda(torch.flatten)]
if HPARAMS["xinv"]:
    transforms.append(tvtransforms.Lambda(lambda x: 1.0 - x))
transform = tvtransforms.Compose(transforms)
mnist_train = tvdatasets.MNIST(DATASETS_ROOT, download=True, train=True, transform=transform)
mnist_test = tvdatasets.MNIST(DATASETS_ROOT, download=True, train=False, transform=transform)
train_images, train_labels = load_dataset(mnist_train)
test_images, test_labels = load_dataset(mnist_test)

assert len(mnist_train) % HPARAMS["batch_size"] == 0, "Keep evenly divisible"

In [ ]:
# JIT


@partial(jax.jit, static_argnums=(1, 5))
def make_step(params, static, opt_state, xs, ys, opt_update):
    def loss_xs(params):
        model = eqx.combine(params, static)
        logitses, xs_neurons = jax.vmap(model)(xs)
        losses = optax.softmax_cross_entropy_with_integer_labels(logitses, ys)
        return losses.mean(), xs_neurons

    (loss, xs_neurons), grad = jax.value_and_grad(loss_xs, has_aux=True)(params)
    updates, opt_state = opt_update(grad, opt_state, params, xs_neurons=xs_neurons)
    new_params = optax.apply_updates(params, updates)
    return new_params, opt_state, loss


@partial(jax.jit, static_argnums=1)
def compute_accuracy(params, static, xs, ys):
    model = eqx.combine(params, static)
    predictions = jax.vmap(lambda x: jnp.argmax(model(x)[0]))(xs)
    return jnp.mean(predictions == ys)

In [ ]:
# Instantiate

# Network
model = eqx.nn.MLP(**HPARAMS["mlp_kwargs"], activation=jax.nn.tanh, key=jax.random.PRNGKey(0))
model = mlp_wrapper.wrap(model)
key = jax.random.PRNGKey(HPARAMS["seed"])
weight_init = getattr(jax.nn.initializers, HPARAMS["init"])(in_axis=1, out_axis=0)
params, static = eqx.partition(model, eqx.is_inexact_array)
params = initialize.weights_biases(params, weight_init=weight_init, key=key)

# Optimizer
opt = optax.chain(
    optax.clip_by_global_norm(HPARAMS["clip_norm"]),
    jaxlnb.lnb(**HPARAMS["lnb_kwargs"]),
    optax.scale_by_learning_rate(HPARAMS["step_size"]),
)
opt_state = opt.init(params)

In [ ]:
# Training loop

data_loader = torch.utils.data.DataLoader(
    dataset=mnist_train,
    batch_size=HPARAMS["batch_size"],
    shuffle=True,
    collate_fn=np_collate,
    generator=torch.Generator().manual_seed(HPARAMS["seed"]),
)
num_batches = len(data_loader)

t = 0
for epoch in range(200):
    for i, (xs, ys) in enumerate(data_loader):
        xs, ys = jnp.array(xs), jnp.array(ys)
        params, opt_state, loss = make_step(params, static, opt_state, xs, ys, opt.update)

        if t % 5 == 0:
            loss = loss.item()
            print(f"[{epoch}, {i:02d}/{num_batches}, {t:04d}] loss: {loss:0.5f}")
        t += 1

    train_acc = compute_accuracy(params, static, train_images, train_labels).item()
    test_acc = compute_accuracy(params, static, test_images, test_labels).item()
    print(f"[{epoch}, -----, {t:04d}] Train Acc: {train_acc:0.4f}    Test Acc: {test_acc:0.4f}")